# Microsoft Fabric Notebook Cheatsheet

A runnable reference of common patterns for working with notebooks in Microsoft Fabric — covering Spark/Pandas interop, file I/O, Delta Lake, transformations, SQL, parameters, `notebookutils`, and performance.

Every demo cell creates its own sample data so the notebook runs top to bottom with no external files. Real-world lakehouse paths appear as commented templates next to each demo.

## How to use

1. Download `NB_Fabric_Cheatsheet.ipynb`.
2. In your Fabric workspace: **New → Import notebook → Upload**.
3. Attach any lakehouse (the demos don't depend on its contents).
4. Run cells in order. The final cell drops the demo tables for cleanup.

Built for the **Synapse PySpark** runtime (the Fabric default). `notebookutils` cells are mostly commented — uncomment to execute against your own tenant.

## Contents

| # | Section | Covers |
|---|---|---|
| 1 | [Setup & imports](#setup) | Standard imports, runtime info |
| 2 | [DataFrame conversions](#conversions) | Spark ↔ Pandas ↔ Pandas-on-Spark |
| 3 | [Reading files](#read) | CSV, Parquet, Excel, JSON (pandas + Spark) |
| 4 | [Writing files](#write) | Pandas writers, Spark writers, single-file output |
| 5 | [Lakehouse paths](#paths) | Relative / mount / ABFSS, `notebookutils.fs` |
| 6 | [Delta tables](#delta) | `saveAsTable`, `save`, append, CTAS |
| 7 | [Delta operations](#delta-ops) | MERGE, OPTIMIZE, VACUUM, time travel, schema evolution |
| 8 | [Cross-lakehouse access](#cross) | ABFSS + three-part naming |
| 9 | [Transformations](#transforms) | `when`/`otherwise`, rename, cast, dedupe, null handling, group/agg |
| 10 | [SQL in notebooks](#sql) | `%%sql` magic vs `spark.sql()`, temp views |
| 11 | [Parameters](#params) | Parameter cell pattern for pipeline invocation |
| 12 | [`notebookutils`](#nbutils) | `notebook.run`, `credentials`, `runtime`, `lakehouse` |
| 13 | [Logging & error handling](#logging) | Pipeline-friendly try/except pattern |
| 14 | [Performance tips](#perf) | Partitioning, broadcast, cache, V-Order |
| 15 | [Inspect / debug utilities](#utils) | Schema, count, info, dataframe-type check |


---
<a id="setup"></a>
## 1. Setup & imports


#### **Imports.** Brings in pandas, the PySpark functions and types modules, the Delta Lake table API, and the standard date/time types. `spark` and `display()` are pre-defined by the Fabric runtime — no import needed. Running `spark.version` confirms the kernel is alive and shows which Spark build you're on.

In [2]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType, TimestampType
from delta.tables import DeltaTable
from datetime import datetime, date

# spark and display() are provided by the Fabric runtime
print("Spark version:", spark.version)


StatementMeta(, c4dbb66e-9dc4-478a-bd24-7c64d29e5f99, 7, Finished, Available, Finished, False)

Spark version: 3.5.5.5.4.20260403.6


---
<a id="conversions"></a>
## 2. DataFrame conversions (Spark ↔ Pandas)

Spark DataFrames are distributed across executors; Pandas DataFrames live in driver memory. Convert carefully — `toPandas()` collects *all* rows to the driver and will OOM on large data.


#### **Spark → Pandas.** `toPandas()` pulls every row of a Spark DataFrame back to the driver and returns a pandas DataFrame. Safe for small result sets (aggregates, samples) — never for raw fact tables.

In [3]:
spark_df = spark.createDataFrame(
    [(1, "apple", 1.20), (2, "banana", 0.50), (3, "orange", 0.80)],
    ["id", "fruit", "price"],
)

pandas_df = spark_df.toPandas()
display(pandas_df)


StatementMeta(, c4dbb66e-9dc4-478a-bd24-7c64d29e5f99, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4a0725f0-c2c4-48b6-ab51-748aac44ae63)

#### **Pandas → Spark.** `spark.createDataFrame(pdf)` distributes a driver-side pandas DataFrame across the cluster. Schema is inferred from the pandas dtypes.

In [6]:
pdf = pandas_df

sdf = spark.createDataFrame(pdf)
display(sdf)


StatementMeta(, c4dbb66e-9dc4-478a-bd24-7c64d29e5f99, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 788a6383-3cc5-4e5c-9673-3d65b650eb3a)

#### **Pandas-on-Spark.** The `pyspark.pandas` module gives you the pandas API but the data stays distributed on the cluster. Use this when you want pandas-style code without collecting to the driver. `.to_spark()` and `.to_pandas()` convert between flavours.

In [7]:
import pyspark.pandas as ps

psdf = ps.DataFrame({"id": [1, 2, 3], "fruit": ["apple", "banana", "orange"]})
display(psdf.head())

# Convert between the three flavours
sdf_again = psdf.to_spark()    # → Spark DataFrame (distributed)
pdf_again = psdf.to_pandas()   # → Pandas (collects to driver)


StatementMeta(, c4dbb66e-9dc4-478a-bd24-7c64d29e5f99, 13, Finished, Available, Finished, False)

/opt/spark/python/lib/pyspark.zip/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.


ImportError: cannot import name '_builtin_table' from 'pandas.core.common' (/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/pandas/core/common.py)

---
<a id="read"></a>
## 3. Reading files

Two patterns: **pandas** for small files that fit on the driver, **Spark** for anything large or distributed.

In Fabric, when a lakehouse is attached to the notebook, you can use relative paths like `Files/<filename>`. The full path is:

```
abfss://<workspace>@onelake.dfs.fabric.microsoft.com/<lakehouse>.Lakehouse/Files/<filename>
```


#### **Read CSV.** `pd.read_csv` for driver-side reads; `spark.read.csv` for distributed reads. The Spark reader needs `header=true` to use the first row as column names and `inferSchema=true` to auto-detect column types (otherwise everything is a string).

In [8]:
# Pandas
# df = pd.read_csv("/lakehouse/default/Files/my_file.csv")

# Spark — preferred for large files
# sdf = (spark.read
#        .option("header", "true")
#        .option("inferSchema", "true")
#        .csv("Files/my_file.csv"))

# Demo with inline CSV text
from io import StringIO
csv_text = "id,fruit,price\n1,apple,1.2\n2,banana,0.5\n3,orange,0.8"
df = pd.read_csv(StringIO(csv_text))
display(df)


StatementMeta(, c4dbb66e-9dc4-478a-bd24-7c64d29e5f99, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1acafeba-1bcb-4ded-97cf-83871a7cf576)

#### **Read Parquet.** Parquet stores schema with the data, so no header or schema options are needed. Both pandas and Spark read it natively.

In [9]:
# Pandas
# df = pd.read_parquet("/lakehouse/default/Files/my_file.parquet")

# Spark
# sdf = spark.read.parquet("Files/my_file.parquet")

# Demo — round-trip through a temp file
import tempfile, os
tmp = os.path.join(tempfile.gettempdir(), "demo.parquet")
pd.DataFrame({"id": [1, 2], "fruit": ["apple", "banana"]}).to_parquet(tmp)
df = pd.read_parquet(tmp)
display(df)


StatementMeta(, c4dbb66e-9dc4-478a-bd24-7c64d29e5f99, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 876a6c2b-fe69-4a69-bbd0-3586e883e60a)

#### **Read Excel.** `pd.read_excel` handles `.xlsx` via the pre-installed `openpyxl` engine. Use `sheet_name` to target a specific tab and `header` to skip rows above the actual header row (common in business spreadsheets where data starts a few rows down). `sheet_name=None` returns a dict of all sheets.

In [10]:
# Pandas — requires openpyxl (pre-installed in Fabric)
# df = pd.read_excel("/lakehouse/default/Files/my_file.xlsx", sheet_name="Sheet1")

# Skip rows / set header row (e.g. data starts at row 5)
# df = pd.read_excel(file_path, sheet_name="Option A and B", header=5)

# Read all sheets at once → dict of DataFrames
# sheets = pd.read_excel(file_path, sheet_name=None)

# Spark needs the spark-excel package; pandas is usually simpler for Excel.


StatementMeta(, c4dbb66e-9dc4-478a-bd24-7c64d29e5f99, 17, Finished, Available, Finished, False)

#### **Read JSON.** `pd.read_json` for driver-side; `spark.read.json` for distributed. Set `multiline=true` for pretty-printed JSON files (objects spanning multiple lines) — without it, Spark expects one JSON object per line (JSONL).

In [11]:
# Pandas
# df = pd.read_json("/lakehouse/default/Files/my_file.json")

# Spark — handles both single-line and multi-line JSON
# sdf = spark.read.option("multiline", "true").json("Files/my_file.json")

# Demo with inline JSON
import json
demo = [{"id": 1, "fruit": "apple"}, {"id": 2, "fruit": "banana"}]
df = pd.read_json(StringIO(json.dumps(demo)))
display(df)


StatementMeta(, c4dbb66e-9dc4-478a-bd24-7c64d29e5f99, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dfe1f738-4109-4bd8-ae26-21f1543b4986)

---
<a id="write"></a>
## 4. Writing files


#### **Write files.** Pandas writers produce a single file. Spark writers produce a *folder* of part-files (one per partition) — that's normal and intended for distributed reads. Use `coalesce(1)` to force a single output file, but only for small data since it collapses everything to one partition first.

Write modes: `overwrite` replaces, `append` adds rows, `ignore` is a no-op if the target exists, `errorifexists` (default) fails.

In [12]:
# Pandas writers — single file
# df.to_csv("/lakehouse/default/Files/out.csv", index=False)
# df.to_parquet("/lakehouse/default/Files/out.parquet")
# df.to_excel("/lakehouse/default/Files/out.xlsx", index=False)
# df.to_json("/lakehouse/default/Files/out.json", orient="records")

# Spark writers — produces a folder of part files
# (sdf.write
#     .mode("overwrite")          # or "append", "ignore", "errorifexists"
#     .option("header", "true")
#     .csv("Files/out_csv_folder"))

# sdf.write.mode("overwrite").parquet("Files/out_parquet_folder")

# Force a single output file (small data only — collapses to one partition)
# sdf.coalesce(1).write.mode("overwrite").option("header","true").csv("Files/out_single")


StatementMeta(, c4dbb66e-9dc4-478a-bd24-7c64d29e5f99, 19, Finished, Available, Finished, False)

---
<a id="paths"></a>
## 5. Lakehouse paths & `notebookutils.fs`

Path styles you'll see:

| Style | Example | When |
|---|---|---|
| Relative | `Files/raw/data.csv` | Default attached lakehouse |
| Local mount | `/lakehouse/default/Files/raw/data.csv` | Pandas / local Python file I/O |
| Full ABFSS | `abfss://ws@onelake.dfs.fabric.microsoft.com/lh.Lakehouse/Files/raw/data.csv` | Cross-workspace / cross-lakehouse |


#### **File-system helpers.** `notebookutils.fs` is the Fabric replacement for `mssparkutils.fs` (the old name still works). Use it for listing, making directories, copying, moving, removing, and mounting external lakehouses.

In [13]:
# List files in the default lakehouse Files area
# notebookutils.fs.ls("Files/")

# Make a directory
# notebookutils.fs.mkdirs("Files/landing/2026-06")

# Copy / move / remove
# notebookutils.fs.cp("Files/a.csv", "Files/archive/a.csv")
# notebookutils.fs.mv("Files/a.csv", "Files/archive/a.csv")
# notebookutils.fs.rm("Files/old/", recurse=True)

# Read a small text file directly
# text = notebookutils.fs.head("Files/sample.txt", 1024)

# Mount another lakehouse (cross-workspace)
# notebookutils.fs.mount(
#     "abfss://<ws>@onelake.dfs.fabric.microsoft.com/<lh>.Lakehouse",
#     "/other_lh"
# )


StatementMeta(, c4dbb66e-9dc4-478a-bd24-7c64d29e5f99, 20, Finished, Available, Finished, False)

---
<a id="delta"></a>
## 6. Delta tables — create, append, overwrite, CTAS

In a Fabric Lakehouse, **managed tables** live under the `Tables/` area and Spark manages both the data and the metastore registration. Four common ways to write them.


#### **`saveAsTable()` — managed table.** Writes the data *and* registers the table in the metastore so it's queryable by name (`spark.table("...")` or SQL). This is the default pattern for lakehouse tables you want to expose to Power BI / SQL endpoint.

In [14]:
sdf = spark.createDataFrame(
    [(1, "apple"), (2, "banana"), (3, "orange")],
    ["id", "fruit"],
)

(sdf.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dim_fruit_demo"))

display(spark.table("dim_fruit_demo"))


StatementMeta(, c4dbb66e-9dc4-478a-bd24-7c64d29e5f99, 21, Finished, Available, Finished, False)

UnsupportedOperationException: No default context found, please attach a lakehouse before running spark sql queries with partial namespaces.

#### **save() to a path — unmanaged.** Writes Delta files to a specific path with no metastore registration. Use this when you want Delta files under `Files/` (e.g. staging Delta data outside the `Tables/` area) — you read them back by path with `spark.read.format("delta").load(path)`.

In [ ]:
# Useful when you want Delta files in Files/ rather than Tables/
delta_path = "Files/delta/dim_fruit_unmanaged"

(sdf.write
    .format("delta")
    .mode("overwrite")
    .save(delta_path))

# Read it back by path
display(spark.read.format("delta").load(delta_path))


StatementMeta(, c4dbb66e-9dc4-478a-bd24-7c64d29e5f99, -1, Cancelled, , Cancelled, True)

#### **Append rows.** `mode("append")` adds new rows to an existing Delta table without touching the existing data. Schemas must match (or use `mergeSchema` — see section 7).

In [ ]:
new_rows = spark.createDataFrame([(4, "pear"), (5, "kiwi")], ["id", "fruit"])

(new_rows.write
    .format("delta")
    .mode("append")
    .saveAsTable("dim_fruit_demo"))

display(spark.table("dim_fruit_demo"))


StatementMeta(, c4dbb66e-9dc4-478a-bd24-7c64d29e5f99, -1, Cancelled, , Cancelled, True)

#### **CTAS — Create Table As Select.** The SQL equivalent of `saveAsTable`. Two steps: register the Spark DataFrame as a temp view from Python, then run a `CREATE TABLE AS SELECT` in a SQL cell. `%%sql` must be the first line of the cell and the entire cell must be SQL — don't mix Python in.

The cell below registers the view from Python.

In [ ]:
# Register the source DataFrame as a temp view so SQL can see it
sdf.createOrReplaceTempView("df_fruit_view")


StatementMeta(, c4dbb66e-9dc4-478a-bd24-7c64d29e5f99, -1, Cancelled, , Cancelled, True)

Then CTAS in a dedicated SQL cell:

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS dim_fruit_ctas
USING DELTA
AS SELECT * FROM df_fruit_view


StatementMeta(, c4dbb66e-9dc4-478a-bd24-7c64d29e5f99, -1, Cancelled, , Cancelled, True)

####  **Pandas → Delta gotcha.** `.write.saveAsTable(...)` only exists on **Spark** DataFrames. If `df` is a pandas DataFrame you'll get `AttributeError`. Convert first:

```python
spark.createDataFrame(df).write.format("delta").mode("overwrite").saveAsTable("my_table")
```


---
<a id="delta-ops"></a>
## 7. Delta operations — MERGE, OPTIMIZE, VACUUM, time travel


#### **MERGE (upsert).** The standard pattern for SCD-1 and idempotent loads. Matches source rows against the target on a join key — updates when matched, inserts when not. `whenNotMatchedInsertAll()` inserts the full source row; you can also use explicit column maps.

In [ ]:
# Source = incoming changes
updates = spark.createDataFrame(
    [(1, "apple-updated"), (4, "pear"), (6, "mango")],
    ["id", "fruit"],
)

target = DeltaTable.forName(spark, "dim_fruit_demo")

(target.alias("t")
    .merge(updates.alias("s"), "t.id = s.id")
    .whenMatchedUpdate(set={"fruit": "s.fruit"})
    .whenNotMatchedInsertAll()
    .execute())

display(spark.table("dim_fruit_demo").orderBy("id"))


StatementMeta(, c4dbb66e-9dc4-478a-bd24-7c64d29e5f99, -1, Cancelled, , Cancelled, True)

#### **OPTIMIZE & VACUUM.** `OPTIMIZE` compacts small files into larger ones — important for tables hit by frequent appends/merges. Fabric applies V-Order by default for better Power BI read performance. `VACUUM` removes data files no longer referenced by the Delta log (default retention 7 days) — keep that retention high enough to support time travel and concurrent readers.

In [ ]:
%%sql
-- Compact small files (Fabric supports V-Order by default)
OPTIMIZE dim_fruit_demo;

-- Remove files no longer referenced by the log (default 7-day retention)
-- VACUUM dim_fruit_demo RETAIN 168 HOURS;


StatementMeta(, c4dbb66e-9dc4-478a-bd24-7c64d29e5f99, -1, Cancelled, , Cancelled, True)

#### **Time travel.** Delta keeps a version history — you can read the table as it existed at a previous version or timestamp. Useful for audit, debugging bad loads, or recovering from a mistaken overwrite. `DESCRIBE HISTORY` lists every commit.

In [ ]:
# By version number
# spark.read.format("delta").option("versionAsOf", 0).table("dim_fruit_demo")

# By timestamp
# spark.read.format("delta").option("timestampAsOf", "2026-01-01").table("dim_fruit_demo")

# History — every commit, who made it, what operation
display(spark.sql("DESCRIBE HISTORY dim_fruit_demo"))


StatementMeta(, c4dbb66e-9dc4-478a-bd24-7c64d29e5f99, -1, Cancelled, , Cancelled, True)

#### **Schema evolution.** By default Delta rejects writes whose schema doesn't match the target. `mergeSchema` lets appends add new columns; `overwriteSchema` lets `overwrite` mode replace the schema entirely. Use sparingly — silent schema drift is a common source of downstream bugs.

In [ ]:
# Allow new columns when appending
# (new_df.write
#     .format("delta")
#     .mode("append")
#     .option("mergeSchema", "true")
#     .saveAsTable("dim_fruit_demo"))

# Allow overwriting the schema entirely
# (new_df.write
#     .format("delta")
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .saveAsTable("dim_fruit_demo"))


---
<a id="cross"></a>
## 8. Cross-lakehouse access


#### **Cross-lakehouse reads.** Three options when the data you want isn't in the default attached lakehouse: full ABFSS path (works from anywhere), three-part naming (when both lakehouses are attached to the notebook), or the SQL endpoint via JDBC (rarely needed for OneLake-native access — prefer ABFSS).

In [ ]:
# Pattern 1 — full ABFSS path to another lakehouse's Tables area
# other_table = (spark.read
#     .format("delta")
#     .load("abfss://<ws>@onelake.dfs.fabric.microsoft.com/<other_lh>.Lakehouse/Tables/dim_customer"))

# Pattern 2 — both lakehouses attached, three-part name
# other_table = spark.table("other_lh.dim_customer")

# Pattern 3 — read from the SQL endpoint of another lakehouse (via JDBC) is also possible
# but for OneLake-native access, ABFSS is the standard


---
<a id="transforms"></a>
## 9. DataFrame transformations


#### **Demo data.** Reset the working DataFrame for the transformation examples — includes a null `price` and a duplicate row so the null-handling and dedupe demos have something to operate on.

In [ ]:
sdf = spark.createDataFrame(
    [(1, "apple", 1.20),
     (2, "banana", 0.50),
     (3, "orange", 0.80),
     (4, "pear", None),
     (2, "banana", 0.50)],   # duplicate
    ["id", "fruit", "price"],
)
display(sdf)


#### **Conditional column with `when` / `otherwise`.** `F.when(condition, value).otherwise(default)` is the Spark equivalent of `CASE WHEN`. Chain multiple `.when()` calls for multi-branch logic.

In [ ]:
sdf_flag = sdf.withColumn(
    "is_apple",
    F.when(F.col("fruit") == "apple", True).otherwise(False),
)
display(sdf_flag)


#### **Rename column.** `withColumnRenamed(old, new)` returns a new DataFrame with one column renamed. To rename many at once, `df.toDF("a", "b", "c")` replaces all column names positionally.

In [ ]:
display(sdf.withColumnRenamed("fruit", "fruit_name"))

# Rename multiple columns at once (positional)
# new_sdf = sdf.toDF("a", "b", "c")


#### **Cast column types.** `F.col("x").cast("string")` returns the column with a new type. Common types: `string`, `int`, `long`, `double`, `decimal(10,2)`, `boolean`, `date`, `timestamp`.

In [ ]:
display(
    sdf.withColumn("id", F.col("id").cast("string"))
       .withColumn("price", F.col("price").cast("double"))
)


#### **Drop duplicates.** `dropDuplicates()` with no arguments removes rows that are duplicates across *all* columns. Pass a subset list to dedupe on specific key columns only.

In [ ]:
# Spark — all columns
display(sdf.dropDuplicates())

# Spark — specific column(s) as the dedupe key
display(sdf.dropDuplicates(["id"]))

# Pandas equivalents
# pdf.drop_duplicates()
# pdf.drop_duplicates(subset=["brand"])
# pdf.drop_duplicates(subset=["brand", "style"], keep="last")


#### **Drop / fill nulls.** `dropna()` removes rows containing any null; pass `subset` to scope to specific columns. `fillna(value)` replaces nulls — pass a single value, or a dict for per-column defaults.

In [ ]:
# Spark
display(sdf.dropna())                              # any null in any column
# sdf.dropna(subset=["price"])                     # only check specific columns
# sdf.fillna(0, subset=["price"])                  # replace nulls with 0
# sdf.fillna({"price": 0.0, "fruit": "unknown"})   # per-column defaults

# Pandas equivalents
# pdf.dropna()
# pdf.fillna({"price": 0})


#### **Filter, select, aggregate.** The bread and butter — `filter` for row predicates, `select` for column projection (with optional aliases), `groupBy().agg()` for aggregations.

In [ ]:
# Filter rows
display(sdf.filter(F.col("price") > 0.6))

# Select specific columns, alias one
display(sdf.select(F.col("fruit").alias("name"), "price"))

# Group and aggregate
display(sdf.groupBy("fruit").agg(
    F.count("*").alias("n"),
    F.avg("price").alias("avg_price"),
))


#### **Audit / load timestamp columns.** Adding `current_timestamp()` and a literal source-system tag is the standard medallion pattern for Bronze loads — gives you provenance and load lineage for free.

In [ ]:
display(
    sdf.withColumn("ingest_ts", F.current_timestamp())
       .withColumn("source_system", F.lit("demo"))
)


---
<a id="sql"></a>
## 10. SQL in notebooks

Two ways to run SQL: `%%sql` magic (whole cell is SQL, results auto-display) or `spark.sql(...)` from Python (returns a DataFrame you can chain).


#### **`%%sql` magic.** When the cell needs to be pure SQL and you want results displayed automatically. Must be the first line of the cell.

In [ ]:
%%sql
SELECT fruit, COUNT(*) AS n
FROM dim_fruit_demo
GROUP BY fruit
ORDER BY n DESC


#### **`spark.sql()` from Python.** Same SQL, but returns a DataFrame you can pass into further transformations, save to a table, or hand to a function.

In [ ]:
# Returns a DataFrame you can chain
result = spark.sql("""
    SELECT fruit, COUNT(*) AS n
    FROM dim_fruit_demo
    GROUP BY fruit
""")
display(result)


#### **Temp views.** Registering a DataFrame as a temp view exposes it to SQL by name. The view lives only for the current Spark session.

In [ ]:
sdf.createOrReplaceTempView("v_fruit")
display(spark.sql("SELECT * FROM v_fruit WHERE price > 0.6"))


---
<a id="params"></a>
## 11. Parameters (for pipelines)

To accept parameters from a Data Factory pipeline, create a cell and tag it as **parameters** (right-side properties panel → Toggle parameter cell). Variables declared in that cell get overridden at runtime by pipeline values. Always declare defaults so the notebook also runs standalone.


#### **Parameter cell.** Tag this cell as `parameters` via the cell properties panel. The values here are defaults — when the notebook is called from a pipeline, the pipeline's parameter values override them.

In [ ]:
# Parameters (tag this cell as 'parameters' in the cell properties)
source_system   = "demo"
load_date       = "2026-01-01"
target_table    = "dim_fruit_demo"
incremental     = False


#### **Using parameters.** Reference the parameter variables downstream as normal Python — the values are either your declared defaults or whatever the calling pipeline injected.

In [ ]:
print(f"Running for {source_system} on {load_date} → {target_table} (incremental={incremental})")


---
<a id="nbutils"></a>
## 12. `notebookutils` essentials

The Fabric replacement for `mssparkutils`. Common modules: `notebook`, `credentials`, `runtime`, `lakehouse`, `fs` (covered earlier).


#### **Notebook orchestration & secrets.** `notebook.run` calls another notebook and waits for it (with a timeout in seconds); `notebook.exit` returns a value to the caller; `notebook.runMultiple` runs notebooks in parallel as a DAG. `credentials.getSecret` pulls a secret from an Azure Key Vault using the notebook's workspace identity.

In [ ]:
# Run another notebook and capture its exit value
# result = notebookutils.notebook.run("NB_Helper", 1800, {"param1": "value1"})

# Exit a notebook with a value (returned to the caller)
# notebookutils.notebook.exit("success")

# Run multiple notebooks in parallel (DAG)
# notebookutils.notebook.runMultiple([
#     {"path": "NB_Bronze_Load", "params": {"src": "sales"}},
#     {"path": "NB_Bronze_Load", "params": {"src": "customers"}},
# ])

# Get a secret from an Azure Key Vault
# secret = notebookutils.credentials.getSecret(
#     "https://<your-vault>.vault.azure.net/",
#     "secretName",
# )

# Runtime context — workspace, lakehouse, user
# ctx = notebookutils.runtime.context
# print(ctx)


#### **Lakehouse metadata.** `notebookutils.lakehouse` lists lakehouses in the current workspace and resolves a lakehouse by name to its properties (including the ABFSS path) — useful when you need to build cross-lakehouse paths dynamically rather than hard-coding them.

In [ ]:
# List lakehouses in current workspace
# notebookutils.lakehouse.list()

# Get a lakehouse by name → returns properties incl. abfsPath
# lh = notebookutils.lakehouse.get("my_lakehouse")
# print(lh["properties"]["abfsPath"])


---
<a id="logging"></a>
## 13. Logging & error handling


#### **Try / except pattern.** Wrap loads in `try` / `except` so failures surface clearly to the calling pipeline. The key is to re-raise (or call `notebookutils.notebook.exit` with a failure message) — swallowing the exception silently means a failed load looks successful to the orchestrator.

In [ ]:
import logging

logger = logging.getLogger("fabric_nb")
logger.setLevel(logging.INFO)

# Pattern: log + re-raise so the calling pipeline sees the failure
try:
    df = spark.createDataFrame([(1, "apple")], ["id", "fruit"])
    row_count = df.count()
    logger.info(f"Loaded {row_count} rows")
except Exception as e:
    logger.error(f"Load failed: {e}")
    # notebookutils.notebook.exit(f"FAILED: {e}")
    raise


---
<a id="perf"></a>
## 14. Performance tips

Quick reference — the things that actually move the needle on Fabric Spark.

- **Partition wisely.** Aim for ~128–512 MB per partition. `df.rdd.getNumPartitions()` shows current count.
- **`repartition` vs `coalesce`.** `repartition(n)` shuffles to exactly n partitions. `coalesce(n)` only reduces and avoids shuffle.
- **Broadcast small dim tables** (< ~10 MB) in joins.
- **Predicate pushdown.** Filter as early as possible on Delta / Parquet — pushed down to the file scan.
- **Cache only when reused.** Caching once-touched data is wasted memory.
- **V-Order + OPTIMIZE** for read-heavy Delta tables (V-Order is on by default in Fabric).
- **Avoid `toPandas()` / `collect()`** on large DataFrames — that's a driver OOM waiting to happen.


#### **Common performance levers.** `repartition` and `coalesce` control partition count; `F.broadcast()` hints that a small DataFrame should be broadcast to every executor (turns a shuffle join into a map-side join). `cache()` materializes a DataFrame in memory across the cluster — the subsequent `count()` forces evaluation; remember to `unpersist()` when done.

In [ ]:
# Repartition / coalesce
# sdf.repartition(8)          # shuffles to exactly 8 partitions
# sdf.coalesce(1)              # reduces to 1 partition (no shuffle, can be slow)

# Broadcast join hint — small lookup joined to large fact
small = spark.createDataFrame([(1, "A"), (2, "B")], ["id", "label"])
joined = sdf.join(F.broadcast(small), "id", "left")
display(joined)

# Cache + count to materialize, then release
# sdf.cache()
# sdf.count()
# sdf.unpersist()


---
<a id="utils"></a>
## 15. Inspect / debug utilities


#### **Schema, count, sample.** Standard first-look commands: `printSchema()` for the column types, `count()` for row count, `columns` for the list of column names, `limit(n)` for a quick sample. `describe()` and `summary()` give numeric column statistics.

In [ ]:
sdf.printSchema()
print("Rows:", sdf.count())
print("Columns:", sdf.columns)
display(sdf.limit(5))
# sdf.describe().show()      # numeric summary (count/mean/stddev/min/max)
# sdf.summary().show()       # incl. percentiles


#### **Pandas `info()`.** Pandas equivalent — shows column dtypes, non-null counts, and memory usage. Useful when you've converted Spark → Pandas and want a quick structural overview.

In [ ]:
pdf = sdf.toPandas()
print(pdf.info())


#### **Identify DataFrame type.** Helper for code that handles either flavour — checks `isinstance` against pandas, Spark, and pandas-on-Spark DataFrame classes.

In [ ]:
from pyspark.sql import DataFrame as SparkDF

def df_kind(x):
    if isinstance(x, pd.DataFrame):
        return "pandas"
    if isinstance(x, SparkDF):
        return "spark"
    try:
        import pyspark.pandas as ps
        if isinstance(x, ps.DataFrame):
            return "pandas-on-spark"
    except Exception:
        pass
    return type(x).__name__

print(df_kind(sdf))
print(df_kind(pdf))


#### **List DataFrames in memory.** The IPython magic `%who_ls DataFrame` lists pandas DataFrames in the current namespace. The programmatic alternative below works for both pandas and Spark.

In [ ]:
# Magic command — uncomment to run
# %who_ls DataFrame

# Programmatic — works for pandas and Spark
[k for k, v in list(globals().items()) if isinstance(v, (pd.DataFrame, SparkDF))]


### Cleanup (optional)


#### **Drop demo objects.** Uncomment to remove the demo tables and the unmanaged Delta folder created by this notebook.

In [ ]:
# spark.sql("DROP TABLE IF EXISTS dim_fruit_demo")
# spark.sql("DROP TABLE IF EXISTS dim_fruit_ctas")
# notebookutils.fs.rm("Files/delta/dim_fruit_unmanaged", recurse=True)
